In [90]:
import pandas as pd

In [91]:
data_source_path = r"C:\Users\venuv\Documents\ai developer\SentimentAnalysis\archive\imbd_reviews.csv".replace("\\", "/")
print(data_source_path)

C:/Users/venuv/Documents/ai developer/SentimentAnalysis/archive/imbd_reviews.csv


In [92]:
df = pd.read_csv(data_source_path)
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [93]:
#remove all the html tags from the review column 
import re
modified_reviews = df["review"].apply(lambda x: re.sub("<.*?>", "", x))

In [94]:
modified_reviews[modified_reviews.apply(lambda x: x.strip()=="")] #checking for empty spaces
modified_reviews[modified_reviews.isna()] #checking for nan values

Series([], Name: review, dtype: object)

In [95]:
#data transformation
modified_reviews = modified_reviews.str.lower()\
.str.replace("[^\w\s]+", "", regex=True)\
.str.replace(r"\s+", " ")\
.str.strip()\
.str.split()
modified_reviews

0        [one, of, the, other, reviewers, has, mentione...
1        [a, wonderful, little, production, the, filmin...
2        [i, thought, this, was, a, wonderful, way, to,...
3        [basically, theres, a, family, where, a, littl...
4        [petter, matteis, love, in, the, time, of, mon...
                               ...                        
49995    [i, thought, this, movie, did, a, down, right,...
49996    [bad, plot, bad, dialogue, bad, acting, idioti...
49997    [i, am, a, catholic, taught, in, parochial, el...
49998    [im, going, to, have, to, disagree, with, the,...
49999    [no, one, expects, the, star, trek, movies, to...
Name: review, Length: 50000, dtype: object

In [96]:
df["review"] = modified_reviews

In [97]:
df["sentiment"] = df["sentiment"].replace("positive", 1).replace("negative", 0)

C:\Users\venuv\AppData\Local\Temp\ipykernel_27020\3780438693.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["sentiment"] = df["sentiment"].replace("positive", 1).replace("negative", 0)


In [99]:
dataFrame = pd.DataFrame({
    "review": df["review"],
    "sentiment":df["sentiment"]
})
dataFrame.to_csv("archive/cleaned.csv")

In [87]:
from collections import Counter
wordCounts = Counter()
for tokens in df["review"]:
    wordCounts.update(tokens)
vocab_size = 20000
commonWords = wordCounts.most_common(vocab_size)


In [88]:
vocab = {}
index = 0
for token, _ in commonWords:
    if token not in vocab:
        vocab[token] = index 
        index+=1

In [89]:
indexedVectors = []
for tokens in df["review"]:
    vector = []
    for token in tokens:
        if(vocab.get(token, 0)!=0):
            vector.append(vocab.get(token))
    if(len(vector)==0):
        vector = [-1]
    indexedVectors.append(vector)

In [70]:
import random
dimensions = 50
numberOfWords = len(vocab)
embeeding_matrix = [[random.uniform(-0.5, 0.5) for _ in range(dimensions)] for _ in range(numberOfWords)]


In [71]:
hidden_dimensions = 50
wx = [[random.uniform(-0.1, 0.1) for _ in range(dimensions)] for _ in range(hidden_dimensions)]
wh = [[random.uniform(-0.1, 0.1) for _ in range(hidden_dimensions)] for _ in range(hidden_dimensions)]
bh = [0.0]*hidden_dimensions

In [72]:
import math
def tanh(x):
    return math.tanh(x)

In [73]:
def rnn_forward(tokens):
    h_list = []
    x_list = []
    h = [0.0]*hidden_dimensions
    h_list.append(h[:])
    for wordId in tokens:
        if(wordId==-1):
            continue
        xe = embeeding_matrix[wordId]
        x_list.append(xe)
        new_h = [0.0]*hidden_dimensions
        for i in range(hidden_dimensions):
            val = 0
            for j in range(dimensions):
                val+=wx[i][j]*xe[j]
            for j in range(hidden_dimensions):
                val+=wh[i][j]*h[j]
            val+=bh[i]
            val = tanh(val)
            new_h[i] = val
        h = new_h
        h_list.append(h[:])
    
    return h_list, x_list



In [74]:
def sigmoid(x):
    return 1/(1+math.exp(-x))

In [75]:
wy = [random.uniform(-0.1, 0.1) for _ in range(hidden_dimensions)]
by = 0

In [76]:

def predict(tokens):
    h_list, x_list = rnn_forward(tokens)
    h = h_list[-1]
    val = 0
    for w,hx in zip(wy, h):
        val+=(w*hx)
    val+=by
    y_pred = sigmoid(val)
    return y_pred, h_list

In [77]:
lr=0.001

In [78]:
for epoch in range(100):
    total_loss = 0
    correct = 0
    for tokens, y_actual in zip(indexedVectors, df["sentiment"]):
        y_pred, h_list = predict(tokens)
        y_roundoff = 1 if y_pred>0.5 else 0
        if(y_roundoff==y_actual):
            correct+=1
        loss = -(y_actual*math.log(y_pred+1e-9)+(1-y_actual)*math.log(1-y_pred+1e-9))
        total_loss+=loss
        error = y_actual-y_pred

        h_last = h_list[-1]
        #update wy
        for i in range(hidden_dimensions):
            wy[i]+=lr*error*(h_last[i])
        #update by
        by+=lr*error

        dh = [error*wy[i] for i in range(hidden_dimensions)]

        for t in reversed(range(len(tokens))):
            h = h_list[t+1]
            h_prev = h_list[t]
            x = embeeding_matrix[tokens[t]]

            #tanh derivative
            dh_raw = [0.0]*hidden_dimensions
            for i in range(hidden_dimensions):
                dh_raw[i] = dh[i]*(1-h[i]**2)
            
            #update wx
            for i in range(hidden_dimensions):
                for j in range(dimensions):
                    wx[i][j]+=lr*dh_raw[i]*x[j]
            
            #update wh
            for i in range(hidden_dimensions):
                for j in range(hidden_dimensions):
                    wh[i][j]+=lr*dh_raw[i]*h_prev[j]

            #update bias
            for i in range(hidden_dimensions):
                bh[i]+=lr*dh_raw[i]

            #update embeedings
            word_id = tokens[t]
            for j in range(dimensions):
                grad=0
                for i in range(hidden_dimensions):
                    grad+=dh_raw[i]*wx[i][j]
                embeeding_matrix[word_id][j]+=lr*grad
            
            #propagate gradient backward
            new_dh = [0.0]*hidden_dimensions
            for i in range(hidden_dimensions):
                for j in range(hidden_dimensions):
                    new_dh[j]+=dh_raw[i]*wh[i][j]
            
            dh = new_dh
    print(f"epoch: {epoch}, loss: {total_loss/len(indexedVectors)}, accuracy:{correct/len(indexedVectors)}")


KeyboardInterrupt: 

In [39]:
print(type(df["sentiment"][0]))

<class 'str'>
